# MindLens Crisis Model Training

Kaggle T4 x2 notebook for training the MindLens crisis classifier.


In [ ]:
!pip install -q transformers datasets accelerate huggingface_hub scikit-learn

In [ ]:
from huggingface_hub import login, HfApi
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logged into HuggingFace")

In [ ]:
import os
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

BASE_MODEL = "distilbert-base-uncased"
YOUR_HF_USERNAME = "AmiruMallawarachchi"
DATASET_NAME = f"{YOUR_HF_USERNAME}/mindlens-crisis-cleaned"
OUTPUT_DIR = "/kaggle/working/mindlens-crisis"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "precision": precision_score(labels, preds, zero_division=0),
    }

print(f"Loading dataset: {DATASET_NAME}")
ds = load_dataset(DATASET_NAME)
train_ds = ds["train"]
val_ds = ds.get("validation") or ds.get("test") or train_ds.select(range(5000))

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
sample_labels = [train_ds[i]["label"] for i in range(min(100, len(train_ds)))]
num_labels = max(sample_labels) + 1
print(f"Detected num_labels: {num_labels}")

model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=num_labels)

def preprocess(examples):
    text_col = "text" if "text" in examples else "tweet"
    label_col = "label" if "label" in examples else "labels"
    tokens = tokenizer(examples[text_col], truncation=True, padding=False, max_length=256)
    tokens["labels"] = examples[label_col]
    return tokens

train_ds = train_ds.map(preprocess, batched=True)
val_ds = val_ds.map(preprocess, batched=True)

columns_to_remove = [c for c in train_ds.column_names if c not in ["input_ids", "attention_mask", "labels"]]
train_ds = train_ds.remove_columns(columns_to_remove)
val_ds = val_ds.remove_columns(columns_to_remove)

In [ ]:
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="recall",
    greater_is_better=True,
    logging_steps=250,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("\n>>> STARTING CRISIS TRAINING <<<\n")
trainer.train()

metrics = trainer.evaluate()
print("\n" + "=" * 60)
print(f"FINAL RECALL: {metrics['eval_recall']:.4f}")
print(f"FINAL F1:     {metrics['eval_f1']:.4f}")
print(f"FINAL ACC:    {metrics['eval_accuracy']:.4f}")
print("=" * 60)

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

api = HfApi()
repo_id = f"{YOUR_HF_USERNAME}/mindlens-crisis"
api.create_repo(repo_id=repo_id, exist_ok=True)
api.upload_folder(folder_path=OUTPUT_DIR, repo_id=repo_id)
print(f"\n>>> UPLOADED: https://huggingface.co/{repo_id}")